In [1]:
import pandas as pd
import numpy as np

#load file
df = pd.read_csv("0.5_cdc_case_surveillance_sample.csv")

#preview the file
print(df.head())
print(df.shape)
print(df.columns)

  cdc_case_earliest_dt cdc_report_dt pos_spec_dt    onset_dt  \
0           2021-08-11    2021/08/13         NaN  2021/08/18   
1           2021-01-14    2021/02/02         NaN  2021/01/14   
2           2022-01-25           NaN         NaN         NaN   
3           2022-01-30    2022/09/28         NaN         NaN   
4           2022-08-23    2022/08/23         NaN         NaN   

              current_status   sex      age_group race_ethnicity_combined  \
0  Laboratory-confirmed case  Male  10 - 19 Years         Hispanic/Latino   
1  Laboratory-confirmed case  Male  10 - 19 Years         Hispanic/Latino   
2  Laboratory-confirmed case  Male  10 - 19 Years         Hispanic/Latino   
3  Laboratory-confirmed case  Male  10 - 19 Years         Hispanic/Latino   
4  Laboratory-confirmed case  Male  10 - 19 Years         Hispanic/Latino   

   hosp_yn   icu_yn death_yn medcond_yn  
0       No  Missing  Missing    Missing  
1       No  Missing       No    Missing  
2  Missing  Missing  Missi

In [2]:
#date columns to datetime
date_cols = ["cdc_case_earliest_dt", "cdc_report_dt", "pos_spec_dt", "onset_dt"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# check results
print(df[date_cols].dtypes)
print(df[date_cols].isna().sum())

cdc_case_earliest_dt    datetime64[ns]
cdc_report_dt           datetime64[ns]
pos_spec_dt             datetime64[ns]
onset_dt                datetime64[ns]
dtype: object
cdc_case_earliest_dt         0
cdc_report_dt            14886
pos_spec_dt             108631
onset_dt                121492
dtype: int64


In [3]:
#keeping rows with a valid case date
df = df.dropna(subset=["cdc_case_earliest_dt"]).copy()

print(df.shape)
print(df["cdc_case_earliest_dt"].head())

(191662, 12)
0   2021-08-11
1   2021-01-14
2   2022-01-25
3   2022-01-30
4   2022-08-23
Name: cdc_case_earliest_dt, dtype: datetime64[ns]


In [4]:
#creating daily case counts:
daily_cases = (
    df.groupby("cdc_case_earliest_dt")
      .size()
      .reset_index(name="daily_case_count")
      .sort_values("cdc_case_earliest_dt")
      .reset_index(drop=True)
)

print(daily_cases.head(10))
print(daily_cases.tail(10))

  cdc_case_earliest_dt  daily_case_count
0           2020-01-01                 3
1           2020-01-02                 1
2           2020-01-04                 1
3           2020-01-06                 1
4           2020-02-13                 1
5           2020-02-25                 2
6           2020-02-26                 2
7           2020-02-28                 1
8           2020-02-29                 1
9           2020-03-01                 3
     cdc_case_earliest_dt  daily_case_count
1565           2024-06-04                14
1566           2024-06-05                18
1567           2024-06-06                14
1568           2024-06-07                 7
1569           2024-06-08                 8
1570           2024-06-09                 7
1571           2024-06-10                13
1572           2024-06-11                14
1573           2024-06-12                14
1574           2024-06-13                17


In [5]:
#creating daily case changes, base point of comparison previous day
#raw difference:
daily_cases["daily_case_change"] = daily_cases["daily_case_count"].diff()

#percent change:
daily_cases["daily_case_pct_change"] = (
    daily_cases["daily_case_count"].pct_change() * 100
)

print(daily_cases.head(10))

  cdc_case_earliest_dt  daily_case_count  daily_case_change  \
0           2020-01-01                 3                NaN   
1           2020-01-02                 1               -2.0   
2           2020-01-04                 1                0.0   
3           2020-01-06                 1                0.0   
4           2020-02-13                 1                0.0   
5           2020-02-25                 2                1.0   
6           2020-02-26                 2                0.0   
7           2020-02-28                 1               -1.0   
8           2020-02-29                 1                0.0   
9           2020-03-01                 3                2.0   

   daily_case_pct_change  
0                    NaN  
1             -66.666667  
2               0.000000  
3               0.000000  
4               0.000000  
5             100.000000  
6               0.000000  
7             -50.000000  
8               0.000000  
9             200.000000  


In [6]:
#Rolling averages 7 ad 14 days:
#7 day rolling average:
daily_cases["rolling_avg_7day"] = (
    daily_cases["daily_case_count"]
    .rolling(window=7, min_periods=1)
    .mean()
)

#14 day rolling average:
daily_cases["rolling_avg_14day"] = (
    daily_cases["daily_case_count"]
    .rolling(window=14, min_periods=1)
    .mean()
)

print(daily_cases.head(15))

   cdc_case_earliest_dt  daily_case_count  daily_case_change  \
0            2020-01-01                 3                NaN   
1            2020-01-02                 1               -2.0   
2            2020-01-04                 1                0.0   
3            2020-01-06                 1                0.0   
4            2020-02-13                 1                0.0   
5            2020-02-25                 2                1.0   
6            2020-02-26                 2                0.0   
7            2020-02-28                 1               -1.0   
8            2020-02-29                 1                0.0   
9            2020-03-01                 3                2.0   
10           2020-03-02                 1               -2.0   
11           2020-03-03                 2                1.0   
12           2020-03-04                 2                0.0   
13           2020-03-05                 1               -1.0   
14           2020-03-06                 

In [7]:
#normalized case rates relative to the sample size:
#total number of sampled cases
total_sample_cases = len(df)
print("Total sampled cases:", total_sample_cases)

#normalized daily case rate per 1,000 sampled cases
daily_cases["normalized_case_rate_per_1000"] = (
    daily_cases["daily_case_count"] / total_sample_cases
) * 1000

#normalized daily case rate per 10,000 sampled cases
daily_cases["normalized_case_rate_per_10000"] = (
    daily_cases["daily_case_count"] / total_sample_cases
) * 10000

print(daily_cases.head(10))

Total sampled cases: 191662
  cdc_case_earliest_dt  daily_case_count  daily_case_change  \
0           2020-01-01                 3                NaN   
1           2020-01-02                 1               -2.0   
2           2020-01-04                 1                0.0   
3           2020-01-06                 1                0.0   
4           2020-02-13                 1                0.0   
5           2020-02-25                 2                1.0   
6           2020-02-26                 2                0.0   
7           2020-02-28                 1               -1.0   
8           2020-02-29                 1                0.0   
9           2020-03-01                 3                2.0   

   daily_case_pct_change  rolling_avg_7day  rolling_avg_14day  \
0                    NaN          3.000000           3.000000   
1             -66.666667          2.000000           2.000000   
2               0.000000          1.666667           1.666667   
3               0.

## Derived Features:
Daily case count: Number of cases recorded per day.
Daily case change: Difference in case counts between consecutive days.
Daily case percent change: Relative change in case counts compared to the previous day.
7-day rolling average: Smoothed average of daily cases over a 7-day window to capture short-term trends.
14-day rolling average: Longer-term smoothing to highlight broader trends.
Normalized case rate: Daily case counts scaled relative to the total number of records in the sampled dataset to allow consistent comparison across time. Rates were expressed per 1,000 and per 10,000 cases to present the data on a more interpretable scale.

In [8]:
daily_cases.to_csv("derived_daily_features_sample.csv", index=False)
print("Saved file: derived_daily_features_sample.csv")

Saved file: derived_daily_features_sample.csv
